In [1]:
import os
# Use the repository root as the working directory, whether this notebook is
# launched from the repo root or from the notebooks/ folder.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')


# Notebook 08: Feature Ablation (UGRansome only)

Per EXPERIMENT_CONFIG.md:

**Tier A** , raw flow features only  
Drop: Family, Threats, Clusters, USD (and their OHE encodings)  
Keep: Time, Netflow_Bytes, Port, Protocol, Flag, IPaddress

**Tier B** , incremental add-back  
Start from Tier A. Add back Family → Threats → Clusters → USD. Record metrics at each step.

In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, matthews_corrcoef, balanced_accuracy_score,
    average_precision_score, confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

Libraries loaded.


In [3]:
train_df = pd.read_csv('data/processed/ugr_train.csv')
test_df  = pd.read_csv('data/processed/ugr_test.csv')

X_tr_full = train_df.drop(columns=['Prediction'])
y_tr      = train_df['Prediction']
X_te_full = test_df.drop(columns=['Prediction'])
y_te      = test_df['Prediction']

print(f'Full feature set: {X_tr_full.shape[1]} features')
print(X_tr_full.columns.tolist())

Full feature set: 49 features
['Time', 'USD', 'Netflow_Bytes', 'Port', 'Protocol_ICMP', 'Protocol_TCP', 'Protocol_UDP', 'Flag_A', 'Flag_AF', 'Flag_AP', 'Flag_APRSF', 'Flag_APS', 'Flag_APSF', 'Flag_ARF', 'Flag_ASF', 'Flag_R', 'IPaddress_A', 'IPaddress_B', 'IPaddress_C', 'IPaddress_D', 'Family_APT', 'Family_CryptXXX', 'Family_CryptoLocker', 'Family_CryptoLocker2015', 'Family_Cryptohitman', 'Family_DMALocker', 'Family_EDA2', 'Family_Flyper', 'Family_Globe', 'Family_Globev3', 'Family_JigSaw', 'Family_Locky', 'Family_NoobCrypt', 'Family_Razy', 'Family_SamSam', 'Family_TowerWeb', 'Family_WannaCry', 'Threats_Blacklist', 'Threats_Botnet', 'Threats_DoS', 'Threats_NerisBotnet', 'Threats_Port Scanning', 'Threats_SSH', 'Threats_Scan', 'Threats_Spam', 'Threats_UDP Scan', 'Clusters_1', 'Clusters_2', 'Clusters_3']


In [4]:
# Identify feature groups from OHE column names
family_cols   = [c for c in X_tr_full.columns if c.startswith('Family_')]
threats_cols  = [c for c in X_tr_full.columns if c.startswith('Threats_')]
clusters_cols = [c for c in X_tr_full.columns if c.startswith('Clusters_')]
usd_cols      = ['USD']

print('Family cols:', family_cols)
print('Threats cols:', threats_cols)
print('Clusters cols:', clusters_cols)
print('USD cols:', usd_cols)

Family cols: ['Family_APT', 'Family_CryptXXX', 'Family_CryptoLocker', 'Family_CryptoLocker2015', 'Family_Cryptohitman', 'Family_DMALocker', 'Family_EDA2', 'Family_Flyper', 'Family_Globe', 'Family_Globev3', 'Family_JigSaw', 'Family_Locky', 'Family_NoobCrypt', 'Family_Razy', 'Family_SamSam', 'Family_TowerWeb', 'Family_WannaCry']
Threats cols: ['Threats_Blacklist', 'Threats_Botnet', 'Threats_DoS', 'Threats_NerisBotnet', 'Threats_Port Scanning', 'Threats_SSH', 'Threats_Scan', 'Threats_Spam', 'Threats_UDP Scan']
Clusters cols: ['Clusters_1', 'Clusters_2', 'Clusters_3']
USD cols: ['USD']


In [5]:
def eval_rf(X_tr, X_te, y_tr, y_te, label):
    rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    rf.fit(X_tr, y_tr)
    y_pred = rf.predict(X_te)
    y_prob = rf.predict_proba(X_te)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
    return {
        'tier': label,
        'n_features': X_tr.shape[1],
        'f1_macro': round(f1_score(y_te, y_pred, average='macro'), 6),
        'f1_attack': round(f1_score(y_te, y_pred, pos_label=1), 6),
        'f1_benign': round(f1_score(y_te, y_pred, pos_label=0), 6),
        'accuracy':  round(accuracy_score(y_te, y_pred), 6),
        'auc_roc':   round(roc_auc_score(y_te, y_prob), 6),
        'mcc':       round(matthews_corrcoef(y_te, y_pred), 6),
        'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn)
    }

results = []

# Tier A: raw flow only (drop Family, Threats, Clusters, USD)
drop_all = family_cols + threats_cols + clusters_cols + usd_cols
X_tr_A = X_tr_full.drop(columns=drop_all)
X_te_A = X_te_full.drop(columns=drop_all)
print(f'Tier A: {X_tr_A.shape[1]} features , computing ...')
results.append(eval_rf(X_tr_A, X_te_A, y_tr, y_te, 'A_raw_flow_only'))
print(f"  F1={results[-1]['f1_macro']:.6f}")

# Tier B1: A + Family
drop_b1 = threats_cols + clusters_cols + usd_cols
X_tr_B1 = X_tr_full.drop(columns=drop_b1)
X_te_B1 = X_te_full.drop(columns=drop_b1)
print(f'Tier B1 (A + Family): {X_tr_B1.shape[1]} features , computing ...')
results.append(eval_rf(X_tr_B1, X_te_B1, y_tr, y_te, 'B1_A_plus_Family'))
print(f"  F1={results[-1]['f1_macro']:.6f}")

# Tier B2: A + Family + Threats
drop_b2 = clusters_cols + usd_cols
X_tr_B2 = X_tr_full.drop(columns=drop_b2)
X_te_B2 = X_te_full.drop(columns=drop_b2)
print(f'Tier B2 (A + Family + Threats): {X_tr_B2.shape[1]} features , computing ...')
results.append(eval_rf(X_tr_B2, X_te_B2, y_tr, y_te, 'B2_A_plus_Family_Threats'))
print(f"  F1={results[-1]['f1_macro']:.6f}")

# Tier B3: A + Family + Threats + Clusters
drop_b3 = usd_cols
X_tr_B3 = X_tr_full.drop(columns=drop_b3)
X_te_B3 = X_te_full.drop(columns=drop_b3)
print(f'Tier B3 (A + Family + Threats + Clusters): {X_tr_B3.shape[1]} features , computing ...')
results.append(eval_rf(X_tr_B3, X_te_B3, y_tr, y_te, 'B3_A_plus_Family_Threats_Clusters'))
print(f"  F1={results[-1]['f1_macro']:.6f}")

# Tier B4: full (A + Family + Threats + Clusters + USD)
print(f'Tier B4 (full, all features): {X_tr_full.shape[1]} features , computing ...')
results.append(eval_rf(X_tr_full, X_te_full, y_tr, y_te, 'B4_full'))
print(f"  F1={results[-1]['f1_macro']:.6f}")

df_ablation = pd.DataFrame(results)
df_ablation.to_csv('results/ablation_ugr.csv', index=False)
print('\nSaved results/ablation_ugr.csv')
df_ablation[['tier','n_features','f1_macro','f1_attack','f1_benign','accuracy','auc_roc','mcc']]

Tier A: 19 features , computing ...


  F1=0.967663
Tier B1 (A + Family): 36 features , computing ...


  F1=0.979017
Tier B2 (A + Family + Threats): 45 features , computing ...


  F1=0.987843
Tier B3 (A + Family + Threats + Clusters): 48 features , computing ...


  F1=0.989032
Tier B4 (full, all features): 49 features , computing ...


  F1=0.991080

Saved results/ablation_ugr.csv


,tier,n_features,f1_macro,f1_attack,f1_benign,accuracy,auc_roc,mcc
0,A_raw_flow_only,19,0.967663,0.950012,0.985314,0.977298,0.986362,0.935339
1,B1_A_plus_Family,36,0.979017,0.967679,0.990355,0.985144,0.998901,0.958088
2,B2_A_plus_Family_Threats,45,0.987843,0.981238,0.994448,0.991431,0.999716,0.975688
3,B3_A_plus_Family_Threats_Clusters,48,0.989032,0.983076,0.994988,0.992266,0.999773,0.978068
4,B4_full,49,0.991080,0.986235,0.995926,0.993712,0.999857,0.982163


In [6]:
print('Notebook 08 complete.')

Notebook 08 complete.
